In [0]:
%pip install -r ./dbt_requirements.txt

In [0]:
from pathlib import Path
import base64
import os
import shutil
import subprocess
import tempfile
import yaml

current_path = Path.cwd().resolve()

search_locations = [current_path, *current_path.parents]

dbt_project_dir = next(
    (
        location / "sales_pipeline"
        for location in search_locations
        if (location / "sales_pipeline" / "dbt_project.yml").exists()
    ),
    None,
)

if dbt_project_dir is None:
    raise FileNotFoundError(
        "Could not find sales_pipeline/dbt_project.yml. "
        f"Notebook working directory: {current_path}"
    )

print(f"Notebook directory: {current_path}")
print(f"dbt project found: {dbt_project_dir}")
print(f"dbt_project.yml exists: {(dbt_project_dir / 'dbt_project.yml').exists()}")

In [0]:
# Clean up a previous temporary profile if this cell is rerun.
if "dbt_temp_context" in globals():
    dbt_temp_context.cleanup()

dbt_temp_context = tempfile.TemporaryDirectory(prefix="lake_house_dbt_")
dbt_profiles_dir = Path(dbt_temp_context.name)
dbt_profiles_dir.chmod(0o700)

# Retrieve the encrypted private key without printing it.
private_key_b64 = dbutils.secrets.get(
    scope="lakehouse",
    key="snowflake-private-key-b64"
).strip()

try:
    private_key_pem = base64.b64decode(
        private_key_b64,
        validate=True
    )
except Exception as exc:
    raise ValueError(
        "The Snowflake private-key secret is not valid Base64."
    ) from exc

if not private_key_pem.startswith(b"-----BEGIN"):
    raise ValueError("Decoded secret does not contain a valid PEM beginning.")

if b"-----END" not in private_key_pem:
    raise ValueError("Decoded secret does not contain a valid PEM ending.")

# Write the encrypted key only to temporary driver storage.
private_key_path = dbt_profiles_dir / "snowflake_key.p8"
private_key_path.write_bytes(private_key_pem)
private_key_path.chmod(0o600)

dbt_profile = {
    "sales_pipeline": {
        "target": "databricks",
        "outputs": {
            "databricks": {
                "type": "snowflake",
                "account": dbutils.secrets.get(
                    "lakehouse", "snowflake-account"
                ),
                "user": dbutils.secrets.get(
                    "lakehouse", "snowflake-user"
                ),
                "role": dbutils.secrets.get(
                    "lakehouse", "snowflake-role"
                ),
                "database": dbutils.secrets.get(
                    "lakehouse", "snowflake-database"
                ),
                "warehouse": dbutils.secrets.get(
                    "lakehouse", "snowflake-warehouse"
                ),
                "schema": dbutils.secrets.get(
                    "lakehouse", "snowflake-schema"
                ),
                "threads": 4,
                "private_key_path": str(private_key_path),
                "private_key_passphrase": dbutils.secrets.get(
                    "lakehouse",
                    "snowflake-private-key-passphrase"
                ),
            }
        },
    }
}

profiles_path = dbt_profiles_dir / "profiles.yml"

with profiles_path.open("w", encoding="utf-8") as profile_file:
    yaml.safe_dump(
        dbt_profile,
        profile_file,
        sort_keys=False
    )

profiles_path.chmod(0o600)
os.environ["DBT_PROFILES_DIR"] = str(dbt_profiles_dir)

print("Temporary dbt profile created successfully.")
print(f"dbt project: {dbt_project_dir}")
print(f"Temporary profiles directory: {dbt_profiles_dir}")
print("No secret values were displayed.")

In [0]:
def run_dbt(command_arguments):
    command = ["dbt", *command_arguments]

    environment = os.environ.copy()
    environment["DBT_PROFILES_DIR"] = str(dbt_profiles_dir)

    print(f"\nRunning: dbt {' '.join(command_arguments)}")

    result = subprocess.run(
        command,
        cwd=str(dbt_project_dir),
        env=environment,
        text=True,
        capture_output=True,
    )

    if result.stdout:
        print(result.stdout)

    if result.stderr:
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(
            f"dbt {' '.join(command_arguments)} failed "
            f"with exit code {result.returncode}."
        )

    return result


run_dbt(["deps"])
run_dbt(["debug"])

In [0]:
# Display the models dbt recognizes in this project.
run_dbt(["ls", "--resource-type", "model"])

# Compile the models into executable Snowflake SQL without building them.
run_dbt(["compile"])

In [0]:
dbt_build_result = run_dbt([
    "build",
    "--select",
    "+path:models/marts"
])

In [0]:
dbt_temp_context.cleanup()
os.environ.pop("DBT_PROFILES_DIR", None)

print("Temporary RSA key and dbt profile removed successfully.")